# ファイル転送機能の検証 (File Transfer Verification)

このノートブックは、Google Colab (ブラウザ) および VS Code 拡張機能環境におけるファイルのアップロード・ダウンロード機能を検証するために使用します。

This notebook is used to verify file upload/download capabilities in Google Colab (Browser) and VS Code extension environments.

## 1. `google.colab.files.upload()` の検証

**期待される動作:**
*   **Colab (Browser):** ブラウザの下部にファイル選択ボタンが表示されます。
*   **VS Code:** 多くの実装でファイル選択ダイアログが表示されるか、入力フィールドが表示されますが、環境によっては非推奨または動作しない場合があります。

In [ ]:
from google.colab import files

print("google.colab.files.upload() を実行します...")
try:
    uploaded = files.upload()

    for fn in uploaded.keys():
      print('User uploaded file "{name}" with length {length} bytes'.format(
          name=fn, length=len(uploaded[fn])))
except Exception as e:
    print(f"google.colab.files.upload failed: {e}")

## 2. `ipywidgets.FileUpload` の検証 (推奨)

**検証手順:**
1.  下のセルの実行ボタンを押します。
2.  「Upload」ボタンが表示されるか確認してください。(表示されない場合はWidget非対応環境です)
3.  ボタンをクリックして適当なファイル(例: 小さなテキストファイル)を選択してください。
4.  **確認:** セルの出力に `[Success] File '...' saved to disk.` と表示され、ファイル情報 (`ls -l`) が出力されれば成功です。

**VS Codeでの注意点:**
*   ボタンを押しても反応がない場合や、エラーが出る場合は、この方法は使用できません。
*   その場合、VS Codeの左側の「エクスプローラー」パネルにファイルをドラッグ＆ドロップしてアップロードする方法が確実です。

In [ ]:
import ipywidgets as widgets
from IPython.display import display
import os

print("--- ipywidgets.FileUpload Verification ---")

uploader = widgets.FileUpload(
    accept='',  # Accepted file extension
    multiple=False
)

display(uploader)

def on_upload_change(change):
    # This callback runs when the value changes (i.e. file is selected)
    if not change['new']:
        return

    print("\n[Event] Change detected in widget value.")
    
    # Handle different ipywidgets versions (dict vs list/tuple)
    # Most Colab/VSCode environments now return a list of dicts or a tuple of dicts
    files_data = []
    new_value = change['new']
    
    if isinstance(new_value, dict):
        # Older version: {filename: {content: b'', ...}}
        for name, info in new_value.items():
            files_data.append({'name': name, 'content': info['content']})
    elif isinstance(new_value, (list, tuple)):
        # Newer version: [{'name': 'foo.txt', 'content': b'', ...}, ...]
        files_data = new_value

    if not files_data:
        print("[Error] No file data extracted.")
        return

    for item in files_data:
        fname = item.get('name', 'uploaded_file')
        content = item.get('content', b'')
        
        # Ensure content is bytes
        if hasattr(content, 'tobytes'):
            content = content.tobytes()
            
        print(f"[Action] Saving '{fname}' ({len(content)} bytes)...")
        
        try:
            with open(fname, 'wb') as f:
                f.write(content)
            print(f"[Success] File '{fname}' saved to disk.")
            
            # Verify file existence on disk
            if os.path.exists(fname):
                print(f"[Verification] File '{fname}' exists. Size: {os.path.getsize(fname)} bytes.")
                print(f"[Command] ls -l {fname}:")
                os.system(f"ls -l \"{fname}\"")
            else:
                print(f"[Error] File '{fname}' NOT found on disk after write.")
                
        except Exception as e:
            print(f"[Error] Failed to save file: {e}")

uploader.observe(on_upload_change, names='value')
print("Please click the 'Upload' button above and select a file to test.")

## 3. `google.colab.files.download()` の検証

**期待される動作:**
*   **Colab (Browser):** ブラウザのダウンロード機能によりファイルがローカルに保存されます。
*   **VS Code:** 動作しない場合が多いです。自動的にダウンロードされないため、エラーになるか無視されます。

In [ ]:
# テスト用ファイルの作成
with open('test_download.txt', 'w') as f:
  f.write('This is a test file for download verification.')

print("test_download.txt を作成しました。ダウンロードを開始します...")
try:
    files.download('test_download.txt')
except Exception as e:
    print(f"Download failed as expected or error: {e}")

## 4. `IPython.display.FileLink` の検証 (VS Code推奨)

**期待される動作:**
*   **Colab (Browser):** クリック可能なリンクが表示され、クリックするとダウンロードまたは表示されます。
*   **VS Code:** クリック可能なリンクが表示されますが、リモートファイルへのパス解決は環境依存です。

In [ ]:
from IPython.display import FileLink
print("Click the link below to download:")
display(FileLink('test_download.txt'))